In [1]:
!pip install git+https://github.com/huggingface/parler-tts.git
!pip install soundfile transformers torch

  Cloning https://github.com/huggingface/parler-tts.git to /tmp/pip-req-build-h952hxzx
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/parler-tts.git /tmp/pip-req-build-h952hxzx
  Resolved https://github.com/huggingface/parler-tts.git to commit d108732cd57788ec86bc857d99a6cabd66663d68
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/descriptinc/audiotools to /tmp/pip-install-92y9r2zy/descript-audiotools_39ab060847f748e3b288fcda66e7c025
  Running command git clone --filter=blob:none --quiet https://github.com/descriptinc/audiotools /tmp/pip-install-92y9r2zy/descript-audiotools_39ab060847f748e3b288fcda66e7c025
  Resolved https://github.com/descriptinc/audiotools to commit 348ebf2034ce24e2a91a553e3171cb00c0c71678
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) .

In [2]:
import torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

device = "cuda:0" if torch.cuda.is_available() else "cpu"

model = ParlerTTSForConditionalGeneration.from_pretrained("ai4bharat/indic-parler-tts").to(device)
tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-parler-tts")
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)

print("Model loaded on:", device)

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.75G [00:00<?, ?B/s]

  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": false,
  "transformers_version": "4.46.1",
  "use_cache": true,
  "vocab_size": 32128
}

  "_name_or_path": "ylacombe/dac_44khz",
  "architectures": [
    "DacModel"
  ],
  "codebook_dim": 8,
  "codebook_loss_weight": 1.0,
  "codebook_size": 1024,
  "commitment_loss_weight": 0.25,
  "decoder_hidden_si

generation_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Model loaded on: cpu


In [3]:
import soundfile as sf
from IPython.display import Audio, display

# ---- CHANGE THESE ----
prompt = "नमस्ते, मेरा नाम दिव्या है। आज का मौसम बहुत सुहाना है।"

description = "Divya speaks in a slightly expressive tone at a moderate pace with a high quality, close-sounding recording and no background noise."
# ----------------------

desc_ids = description_tokenizer(description, return_tensors="pt").to(device)
prompt_ids = tokenizer(prompt, return_tensors="pt").to(device)

generation = model.generate(
    input_ids=desc_ids.input_ids,
    attention_mask=desc_ids.attention_mask,
    prompt_input_ids=prompt_ids.input_ids,
    prompt_attention_mask=prompt_ids.attention_mask,
)

audio = generation.cpu().numpy().squeeze()
sf.write("output.wav", audio, model.config.sampling_rate)
display(Audio("output.wav"))

In [ ]:
voices = [
    ("Leela",  "Leela speaks in a high-pitched, fast-paced, and cheerful tone. The recording is very high quality with no background noise."),
    ("Divya",  "Divya's voice is monotone yet slightly fast in delivery, with a very close recording that almost has no background noise."),
    ("Rohit",  "Rohit speaks with a moderate pitch and natural pace. The recording is clear and close-sounding."),
    ("Bikram", "Bikram speaks with a higher pitch and fast pace, conveying urgency. The recording is clear and intimate."),
    ("Anjali", "Anjali speaks with a high pitch at a normal pace in a clear, close-sounding environment. Her neutral tone is captured with excellent audio quality."),
    ("Aditi",  "Aditi speaks slowly with a high pitch and expressive tone. The recording is clear, showcasing her energetic and emotive voice."),
]

test_text ="""नदी के किनारे एक छोटा सा गाँव था। वहाँ के लोग बहुत मेहनती और ईमानदार थे।
एक दिन अचानक आसमान में काले बादल छा गए और तेज़ बारिश होने लगी।
बच्चे खुशी से नाचने लगे, बूढ़े बरगद के नीचे बैठ गए, और किसान अपने खेतों की
ओर दौड़ पड़े। उस रात पूरे गाँव में दीये जले और सबने मिलकर खाना खाया।"""

for name, desc in voices:
    desc_ids = description_tokenizer(desc, return_tensors="pt").to(device)
    prompt_ids = tokenizer(test_text, return_tensors="pt").to(device)
    gen = model.generate(
        input_ids=desc_ids.input_ids,
        attention_mask=desc_ids.attention_mask,
        prompt_input_ids=prompt_ids.input_ids,
        prompt_attention_mask=prompt_ids.attention_mask,
    )
    audio = gen.cpu().numpy().squeeze()
    fname = f"{name}.wav"
    sf.write(fname, audio, model.config.sampling_rate)
    print(f"\n--- {name} ---")
    display(Audio(fname))


--- Leela ---



--- Divya ---



--- Rohit ---


In [ ]:
# Cell 1 - install
!pip install git+https://github.com/ai4bharat/IndicF5.git

# Cell 2 - generate with a reference voice
from transformers import AutoModel
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

model = AutoModel.from_pretrained("ai4bharat/IndicF5", trust_remote_code=True)

# you need a reference .wav — download one of their sample prompts
import urllib.request
urllib.request.urlretrieve(
    "https://huggingface.co/ai4bharat/IndicF5/resolve/main/prompts/PAN_F_HAPPY_00001.wav",
    "ref.wav"
)

text = "नमस्ते! आज का दिन बहुत खास है। मैं आपसे कुछ ज़रूरी बात करना चाहता हूँ।"
ref_text = "ਭਹੰਪੀ ਵਿੱਚ ਸਮਾਰਕਾਂ ਦੇ ਭਵਨ ਨਿਰਮਾਣ ਕਲਾ ਦੇ ਵੇਰਵੇ ਗੁੰਝਲਦਾਰ ਅਤੇ ਹੈਰਾਨ ਕਰਨ ਵਾਲੇ ਹਨ।"

audio = model(text, ref_audio_path="ref.wav", ref_text=ref_text)

if audio.dtype == np.int16:
    audio = audio.astype(np.float32) / 32768.0

sf.write("indicf5_out.wav", np.array(audio, dtype=np.float32), samplerate=24000)
display(Audio("indicf5_out.wav"))